# Ising Trace Diagnostics

Run `just example ising_1d` from the repository root first. The example writes `target/ising_1d_trace.csv`, which this notebook reads with Polars for trace inspection and acceptance statistics.

In Colab or another external notebook runtime, install `polars` and `matplotlib` if needed, then upload or mount the generated CSV. Set `MCMC_TRACE_PATH` to an explicit CSV path, or set `MCMC_REPO_ROOT` when the runtime does not start in the repository.

In [ ]:
import csv
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

fixed_columns = ("chain_id", "step", "accepted", "proposed", "log_prob")
required_observables = {"energy", "magnetization"}
configured_root = os.environ.get("MCMC_REPO_ROOT")
repo_root = (Path(configured_root).expanduser() if configured_root else Path.cwd()).resolve()
configured_trace = os.environ.get("MCMC_TRACE_PATH")
trace_candidates = (
    [Path(configured_trace).expanduser()]
    if configured_trace
    else [
        repo_root / "target/ising_1d_trace.csv",
        Path.cwd() / "target/ising_1d_trace.csv",
        Path.cwd().parent / "target/ising_1d_trace.csv",
    ]
)
trace_path = next((candidate.resolve() for candidate in trace_candidates if candidate.is_file()), None)
if trace_path is None:
    checked = ", ".join(str(candidate) for candidate in trace_candidates)
    msg = f"Could not find Ising trace CSV. Checked {checked}. Run `just notebook-check` first."
    raise FileNotFoundError(msg)

with trace_path.open(encoding="utf-8", newline="") as trace_stream:
    header = next(csv.reader(trace_stream), [])
if len(header) != len(set(header)):
    msg = "Trace CSV column names must be unique."
    raise ValueError(msg)
if tuple(header[: len(fixed_columns)]) != fixed_columns:
    msg = f"Trace CSV must start with the fixed columns {fixed_columns}."
    raise ValueError(msg)
observable_columns = header[len(fixed_columns) :]
missing_observables = required_observables.difference(observable_columns)
if missing_observables:
    missing = ", ".join(sorted(missing_observables))
    msg = f"Trace CSV is missing required observables: {missing}."
    raise ValueError(msg)

schema_overrides = {
    "chain_id": pl.UInt64,
    "step": pl.UInt64,
    "accepted": pl.Boolean,
    "proposed": pl.Boolean,
    "log_prob": pl.Float64,
    **dict.fromkeys(observable_columns, pl.Float64),
}
trace = pl.read_csv(trace_path, schema_overrides=schema_overrides)
if trace.is_empty():
    msg = "Trace CSV must contain at least one row."
    raise ValueError(msg)
if sum(trace.null_count().row(0)):
    msg = "Trace CSV must not contain null values."
    raise ValueError(msg)
if (trace["accepted"] & ~trace["proposed"]).any():
    msg = "Every accepted step must have a concrete proposal."
    raise ValueError(msg)
for column in ("log_prob", *observable_columns):
    if not trace[column].is_finite().all():
        msg = f"Trace column {column!r} must contain only finite values."
        raise ValueError(msg)

key_columns = ["chain_id", "step"]
trace_keys = trace.select(key_columns)
if trace_keys.is_duplicated().any():
    msg = "Each (chain_id, step) pair must be unique."
    raise ValueError(msg)
if not trace_keys.equals(trace.sort(key_columns).select(key_columns)):
    msg = "Trace rows must be sorted by chain_id and step."
    raise ValueError(msg)
trace.head()

In [ ]:
summary = (
    trace.group_by("chain_id")
    .agg(
        pl.len().alias("steps"),
        pl.col("accepted").sum().alias("accepted"),
        pl.col("proposed").sum().alias("proposed"),
        pl.col("energy").mean().alias("mean_energy"),
        pl.col("magnetization").mean().alias("mean_magnetization"),
    )
    .with_columns(
        (pl.col("accepted") / pl.col("steps")).alias("step_acceptance_rate"),
        pl.when(pl.col("proposed") > 0).then(pl.col("accepted") / pl.col("proposed")).otherwise(None).alias("proposal_acceptance_rate"),
    )
    .sort("chain_id")
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for chain_trace in trace.partition_by("chain_id", maintain_order=True):
    chain_id = chain_trace["chain_id"][0]
    ax.plot(chain_trace["step"], chain_trace["energy"], label=f"chain {chain_id}")
ax.set_title("Open-boundary 1-D Ising energy trace")
ax.set_xlabel("step")
ax.set_ylabel("energy")
if trace["chain_id"].n_unique() > 1:
    ax.legend()
fig.tight_layout()
figure_output = trace_path.parent / "notebooks/ising_energy_trace.png"
figure_output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(
    figure_output,
    dpi=144,
    metadata={
        "Title": "Open-boundary 1-D Ising energy trace",
        "Description": f"Source={trace_path.name}; rows={trace.height}; chains={trace['chain_id'].n_unique()}",
    },
)
figure_output

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for chain_trace in trace.partition_by("chain_id", maintain_order=True):
    chain_id = chain_trace["chain_id"][0]
    ax.plot(
        chain_trace["step"],
        chain_trace["magnetization"],
        label=f"chain {chain_id}",
    )
ax.set_title("Magnetization trace")
ax.set_xlabel("step")
ax.set_ylabel("magnetization per spin")
if trace["chain_id"].n_unique() > 1:
    ax.legend()
fig.tight_layout()

The exported columns are intentionally plain: `chain_id`, `step`, `accepted`, `proposed`, `log_prob`, then one or more floating-point observables. The loader preserves extra observable columns while requiring the two plotted above. `step_acceptance_rate` matches the Rust `Trace::acceptance_rate` contract (accepted steps divided by all recorded steps); `proposal_acceptance_rate` separately conditions on steps that had a concrete proposal. Later autocorrelation, integrated autocorrelation time, ESS, ESS-rate, and R-hat diagnostics can consume the same validated table without depending on plotting code.

## Multiple chains

The `chain_id` column generalizes this table to several chains. In Rust, give each chain a distinct `ChainId`, accumulate it with its own `TraceRecorder`, then merge the per-chain traces with `Trace::extend` (or concatenate the exported CSVs) before analysis. The `group_by("chain_id")` aggregation above mirrors `Trace::records_for_chain` and `Trace::acceptance_rate`, isolating each chain. Recording two or more `ChainId`s this way produces exactly the multi-chain input that the planned R-hat diagnostic (#74) consumes.